In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

# notebooks에서 실행 중이라면 프로젝트 루트로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\dev\llm_data_analysis-course


In [2]:
import pandas as pd
import numpy as np

customers = pd.read_csv("../data/raw/customers.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
orders = pd.read_csv("../data/raw/orders.csv")
products = pd.read_csv("../data/raw/products.csv")

customers.info()
order_items.info()
orders.info()
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          

In [3]:
# 원본 구조에 Evidence 를 만든다.
raw_data = {"customers" : customers, "order_items": order_items, "products" : products, "orders" : orders}

print(raw_data)

{'customers':      customer_id name gender  age city signup_date
0              1  김수민      F   19   광주  2024-08-14
1              2  김정호      F   32   대구  2025-12-28
2              3  이경수      F   61   성남  2024-08-07
3              4  조영호      F   55   울산  2026-06-08
4              5  이예원      F   19   부산  2024-11-08
..           ...  ...    ...  ...  ...         ...
145          146  김숙자      M   61   성남  2026-02-17
146          147  이정남      M   19   부산  2025-04-08
147          148  오도현      M   29   고양  2026-08-10
148          149  김정자      M   20   부산  2024-12-14
149          150  조미영      M   40   대전  2026-01-29

[150 rows x 6 columns], 'order_items':      order_item_id  order_id  product_id  quantity  unit_price
0                1         1         100         3      102000
1                2         1          87         5       25000
2                3         1           7         3      142000
3                4         1           9         3      193000
4                5 

In [4]:
# 편하게 전체 결과를 확인하는 함수


summary_list = []
for name, frame in raw_data.items():
    # print(name)
    info = {
        "dataset" : name,
        "rows" : len(frame),
        "clumns" : frame.shape[1],
        "missing_values" : int(frame.isna().sum().sum()),
        "duplicated" : int(frame.duplicated().sum()),
    }

summary_list.append(info)

print(summary_list)

[{'dataset': 'orders', 'rows': 300, 'clumns': 5, 'missing_values': 0, 'duplicated': 0}]


In [5]:
from src.preprocessing import (
    compare_shapes,
    preprocess_sales_data,
    validate_relationships,
)
processed_data = preprocess_sales_data(raw_data)

preprocessing_comparison = compare_shapes(
    raw_data,
    processed_data,
)
relationship_checks = validate_relationships(
    processed_data
)

In [8]:
from src.preprocessing import compare_shapes, preprocess_sales_data

processed_data = preprocess_sales_data(raw_data)
preprocess_sales_data = compare_shapes(raw_data, processed_data)

preprocessing_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [12]:
raw_data["orders"].columns
processed_data["orders"].columns

Index(['order_id', 'customer_id', 'order_date', 'payment_method',
       'order_status', 'order_month', 'order_dayofweek'],
      dtype='str')

In [16]:
# 요일별 집계

processed_data["orders"]["order_dayofweek"].value_counts()

order_dayofweek
Saturday     50
Tuesday      47
Monday       47
Wednesday    44
Sunday       43
Friday       35
Thursday     34
Name: count, dtype: int64

In [30]:
key_map = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

7. 전체 행 중복과 PK 중복은 다릅니다

In [31]:
kp_checks = []

for dataset, key in key_map.items():
    frame = processed_data[dataset]

    missing_count = int(frame[key].isna().sum())
    duplicated_count = int(frame[key].duplicated().sum())

    kp_checks.append({

        "dataset" : dataset,
        "key" : key,
        "missing_count" : missing_count,
        "duplicated_count" : duplicated_count,
        "status" : (
            "PASS"
            if missing_count == 0 and duplicated_count == 0
            else "FAIL"
        ),
    })
    

In [32]:
print(kp_checks)

[{'dataset': 'customers', 'key': 'customer_id', 'missing_count': 0, 'duplicated_count': 0, 'status': 'PASS'}, {'dataset': 'products', 'key': 'product_id', 'missing_count': 0, 'duplicated_count': 0, 'status': 'PASS'}, {'dataset': 'orders', 'key': 'order_id', 'missing_count': 0, 'duplicated_count': 0, 'status': 'PASS'}, {'dataset': 'order_items', 'key': 'order_item_id', 'missing_count': 0, 'duplicated_count': 0, 'status': 'PASS'}]


In [ ]:
order_sales = order_items.merge(

    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
        ]
    ],

    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

KeyError: "['line_total'] not in index"

In [34]:
order_sales.head()

,order_item_id,order_id,product_id,quantity,unit_price,customer_id,order_date,order_status,_merge
0,1,1,100,3,102000,123,2026-07-02,completed,both
1,2,1,87,5,25000,123,2026-07-02,completed,both
2,3,1,7,3,142000,123,2026-07-02,completed,both
3,4,1,9,3,193000,123,2026-07-02,completed,both
4,5,2,72,4,189000,77,2025-09-17,cancelled,both


In [37]:
print("병합 전 행 수 : ", len(order_items))
print("병합 후 행 수 : ", len(order_sales))
order_sales["_merge"].value_counts(dropna=False)


병합 전 행 수 :  764
병합 후 행 수 :  764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [38]:
excepted_line_total = order_items["quantity"] * order_items["unit_price"]
excepted_line_total

0      306000
1      125000
2      426000
3      579000
4      756000
        ...  
759    112000
760    696000
761    378000
762    700000
763    160000
Length: 764, dtype: int64

In [39]:
if "line_total" not in order_items.columns:
    order_items["line_total"] = excepted_line_total

    order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
 5   line_total     764 non-null    int64
dtypes: int64(6)
memory usage: 35.9 KB


In [51]:
completed_order_sales = order_sales.loc[
    order_sales["order_status"].eq("completed")

]

print(completed_order_sales.head())
print(completed_order_sales["order_status"].value_counts())

    order_item_id  order_id  product_id  quantity  unit_price  customer_id  \
0               1         1         100         3      102000          123   
1               2         1          87         5       25000          123   
2               3         1           7         3      142000          123   
3               4         1           9         3      193000          123   
12             13         6          83         3       24000           87   

    order_date order_status _merge  
0   2026-07-02    completed   both  
1   2026-07-02    completed   both  
2   2026-07-02    completed   both  
3   2026-07-02    completed   both  
12  2026-05-16    completed   both  
order_status
completed    474
Name: count, dtype: int64


In [46]:
print("전체 주문 건수 : ", len(order_sales))

print("전체 주문 건수(completed) : ", len(completed_order_sales))

print("전체 주문 건수(completed 외) : ", len(order_sales) - len(completed_order_sales))

전체 주문 건수 :  764
전체 주문 건수(completed) :  474
전체 주문 건수(completed 외) :  290


In [52]:
category_sales = (
    completed_order_sales
    .groupby("order_status", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending)
)

#merge로 products 테이블에서 category 컬럼 가져오기

KeyError: "Label(s) ['line_total'] do not exist"